# 📘 Notebook 07: Larger than Memory

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module C: Too Big for One Machine · Notebook 1 of 4 in this module · (7 of 12 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- **Measure** how much memory a piece of data really occupies in Python, and explain why it is far more than the file on disk
- Process a file as a **stream**, in memory that does not grow with the size of the file
- Compute totals, averages and groupings in a **single pass**
- Draw a fair random sample from a stream of unknown length with **reservoir sampling**
- Sort a file that does not fit in memory with an **external merge sort**
- Decide when streaming is worth its cost, and when simply loading the data is the better choice

> **Prerequisites:** Notebooks 01 to 05. Python functions and loops. Generators (`yield`) are introduced here from scratch.
>
> 🔭 **Why this notebook matters:** the usual way to work with data is to load it and then compute. That works until the day the data is larger than the memory of the machine, and then the program does not slow down: it stops. The instinct at that moment is to ask for a bigger machine or a cluster. Very often neither is needed. A laptop can add up a file a hundred times larger than its memory, if it reads the file a piece at a time and remembers only what it must. The same idea, processing data in pieces and combining small results, is also the foundation of every cluster system in the rest of this module.

> 📦 **Libraries used in this notebook.**
>
> - Almost everything is done with Python itself, including **`tracemalloc`**, a standard module that reports how much memory a piece of code used.
> - **PyArrow**, introduced in Notebook 05, appears in Section 6 to read and write files in batches.
>
> The cell below installs what is needed. In Google Colab it is already present and the cell finishes in a moment.

In [ ]:
%pip install -q pyarrow

In [ ]:
import csv
import heapq
import os
import random
import datetime
import sys
import tempfile
import time
import tracemalloc

folder = tempfile.mkdtemp()
print("Files will be written to", folder)

## 1. How much room does data take?

### Intuition first
A file of sales is 20 megabytes on disk. How much memory will it need once loaded? Most people guess "about 20 megabytes". The real answer in Python is five to ten times that, and it is worth seeing why.

In a file, the number `7` is one character: one byte. In Python, `7` is an **object**. It carries its type, a count of how many places refer to it, and the value itself.

In [ ]:
for value in [7, 3.14, "Athens", (7, 3.14, "Athens"), [7, 3.14, "Athens"]]:
    print(f"{str(value):<24} {type(value).__name__:<6} {sys.getsizeof(value):>4} bytes")

A small whole number costs 28 bytes, and a six-letter word 55. A list or tuple adds its own overhead **on top of** the objects it holds, which are counted separately. A row of six values is therefore a few hundred bytes, where its line in the file was perhaps forty.

Let us make a file to experiment with: 400,000 order lines. Notice that the function below **writes each line as soon as it is made**. It never holds the whole dataset, so it could write 400 million lines in the same amount of memory.

In [ ]:
def write_sales(path, lines=400_000, seed=0):
    rng = random.Random(seed)
    cities = ["Athens", "Thessaloniki", "Patras", "Heraklion", "Larissa", "Volos", "Ioannina", "Chania"]
    categories = ["Fiction", "Science", "History", "Children", "Cooking", "Travel", "Art", "Computing"]
    first_day = datetime.date(2023, 1, 1)
    with open(path, "w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(["line_id", "date", "city", "category", "quantity", "revenue"])
        for line_id in range(1, lines + 1):
            day = first_day + datetime.timedelta(days=rng.randint(0, 729))
            quantity = rng.randint(1, 3)
            writer.writerow([line_id, day.isoformat(), rng.choice(cities), rng.choice(categories),
                             quantity, round(quantity * rng.uniform(5, 60), 2)])

sales_path = os.path.join(folder, "sales.csv")
write_sales(sales_path)
file_megabytes = os.path.getsize(sales_path) / 1_000_000
print(f"{file_megabytes:.1f} MB on disk")

Now the measurement. The helper below runs a function and reports its result, the time it took, and the **peak** memory it used while running.

In [ ]:
def measure(function):
    tracemalloc.start()
    started = time.perf_counter()
    result = function()
    elapsed = time.perf_counter() - started
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return result, elapsed, peak / 1_000_000

def total_by_loading():
    with open(sales_path, encoding="utf-8") as file:
        rows = list(csv.reader(file))               # the whole file, as a list of lists
    total = 0.0
    for row in rows[1:]:
        total += float(row[5])
    return total

total, elapsed, peak_loading = measure(total_by_loading)
print(f"Total revenue: {total:,.2f}")
print(f"File on disk:  {file_megabytes:6.1f} MB")
print(f"Peak memory:   {peak_loading:6.1f} MB   ({peak_loading / file_megabytes:.0f} times the size of the file)")

To add up one column we built the entire table in memory, at many times the size of the file, and then threw it away. On this file it does not matter. Imagine the same program given a file of 10 gigabytes on a laptop with 16 gigabytes of memory.

## 2. Streams: one piece at a time

### Intuition first
To count the cars passing on a road, you do not need to stop them all and line them up in a car park. You stand by the road with a counter, and click once for each car that goes by. Your memory holds one number, however long you stand there.

Reading a file works the same way, if we let it. A file object in Python hands out its lines **one at a time**, and the loop below never holds more than the current line.

In [ ]:
def total_by_streaming():
    total = 0.0
    with open(sales_path, encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader)                                # skip the header
        for row in reader:                          # one row at a time
            total += float(row[5])
    return total

total, elapsed, peak_streaming = measure(total_by_streaming)
print(f"Total revenue: {total:,.2f}")
print(f"Peak memory by loading:   {peak_loading:8.2f} MB")
print(f"Peak memory by streaming: {peak_streaming:8.2f} MB")

The same answer, in a tiny fraction of the memory. More important than the size of that number is that it **does not depend on the size of the file**. Whether the file holds four hundred thousand lines or four hundred million, the streaming version needs the same few kilobytes.

### Formal definition
A computation is done in **streaming** fashion, or in **one pass**, when it reads its input sequentially, once, and keeps only a small **state** between one item and the next. Its memory use is $O(1)$ in the length of the input, where loading everything first is $O(n)$.

### Generators: building your own streams
Python has a neat tool for writing code that produces a stream: a function that uses `yield` in place of `return`. Calling it does not run it. It returns a **generator**, which produces one value each time one is asked for, and pauses in between.

In [ ]:
def read_sales(path):
    """Produce the rows of the file one at a time, as dictionaries with proper types."""
    with open(path, encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader)
        for line_id, date, city, category, quantity, revenue in reader:
            yield {"line_id": int(line_id), "date": date, "city": city, "category": category,
                   "quantity": int(quantity), "revenue": float(revenue)}

stream = read_sales(sales_path)
print(stream)                       # nothing has been read yet
print(next(stream))                 # now one row is read
print(next(stream))                 # and now one more

Generators can be chained, each consuming the one before, like stations on an assembly line. No station holds more than the item it is working on.

In [ ]:
def only(rows, **conditions):
    for row in rows:
        if all(row[field] == value for field, value in conditions.items()):
            yield row

def field(rows, name):
    for row in rows:
        yield row[name]

pipeline = field(only(read_sales(sales_path), city="Patras", category="Travel"), "revenue")

result, elapsed, peak = measure(lambda: sum(pipeline))
print(f"Revenue from Travel books in Patras: {result:,.2f}   (peak memory {peak:.2f} MB)")

Read the pipeline from the inside out: read the sales, keep the rows for Patras and Travel, take their revenue, add it up. It is the `SELECT SUM(revenue) ... WHERE ...` of Notebook 01, executed one row at a time.

> ⚠️ **Common pitfalls**
>
> - A generator can be consumed **once**. After `sum(pipeline)` it is empty, and a second `sum(pipeline)` returns 0 without any error. To go through the data again, create the generator again.
> - Writing `list(stream)` "just to have a look". That loads everything, which is the thing we were avoiding. To look at a few items, take a few: `itertools.islice(stream, 5)`.
> - Measuring a streaming program and concluding that it is "slow". It does the same work as the loading version. What it saves is memory.

## 3. What can be computed in one pass?

A great deal. The rule is that the **state** kept between rows must stay small.

**A count, a sum, a minimum, a maximum, an average.** Each needs one or two numbers of state.

In [ ]:
def summary(values):
    count, total, smallest, largest = 0, 0.0, float("inf"), float("-inf")
    for value in values:
        count += 1
        total += value
        smallest = min(smallest, value)
        largest = max(largest, value)
    return {"count": count, "average": round(total / count, 2), "smallest": smallest, "largest": largest}

print(summary(field(read_sales(sales_path), "revenue")))

**A grouping.** The state is one small entry **per group**. Its size depends on the number of groups and not on the number of rows.

In [ ]:
def revenue_by(rows, key):
    totals = {}
    for row in rows:
        totals[row[key]] = totals.get(row[key], 0.0) + row["revenue"]
    return totals

totals, elapsed, peak = measure(lambda: revenue_by(read_sales(sales_path), "category"))
for category, revenue in sorted(totals.items(), key=lambda item: -item[1])[:4]:
    print(f"{category:<10} {revenue:>14,.2f}")
print(f"\n400,000 rows, {len(totals)} groups, peak memory {peak:.2f} MB")

This is a complete `GROUP BY`, in memory proportional to eight categories. It would work unchanged on a file a thousand times longer.

The limit of the technique is visible in the same code. Group by `line_id`, which is different on every row, and the dictionary grows to one entry per row: we are back to holding everything. **Streaming helps when the answer is small, even though the input is large.**

Some questions do not have that property. The **median** revenue is the middle value when all are sorted, and there is no way to find it exactly without, in effect, keeping all the values. For questions like this there are two honest options: accept an approximate answer, or use the disk as extra memory. The next two sections take them in turn.

## 4. A fair sample of a stream

### Intuition first
If we cannot keep every value, we can keep a **sample** of them and compute on that. The difficulty is drawing a fair sample from a stream whose length we do not know in advance. We cannot say *"keep one row in a thousand"*, because we do not know whether the stream holds ten thousand rows or ten billion, and we want exactly, say, 1,000 values at the end.

The solution is known as **reservoir sampling**, and it is a small piece of magic.

- Keep the first $k$ items. That is the reservoir.
- For each later item, number $n$ in the stream: with probability $k / n$, let it **replace** a randomly chosen item of the reservoir. Otherwise, discard it.

At every moment, the reservoir is a perfectly fair sample of everything seen so far.

In [ ]:
def reservoir_sample(stream, k, seed=0):
    rng = random.Random(seed)
    reservoir = []
    for n, item in enumerate(stream, start=1):
        if n <= k:
            reservoir.append(item)
        else:
            position = rng.randrange(n)              # a random position from 0 to n - 1
            if position < k:                         # happens with probability k / n
                reservoir[position] = item
    return reservoir

sample = reservoir_sample(field(read_sales(sales_path), "revenue"), k=2000)
estimate = sorted(sample)[len(sample) // 2]

everything = sorted(field(read_sales(sales_path), "revenue"))          # for checking only: this DOES load every value
exact = everything[len(everything) // 2]

print(f"Median estimated from a sample of {len(sample):,}: {estimate:.2f}")
print(f"Exact median of all {len(everything):,} values:   {exact:.2f}")

Two thousand values, chosen in one pass and in constant memory, give a median very close to the true one.

Is the sample really fair? A simple test: the stream of numbers from 1 to 100,000 has as many values in its first half as in its second. A fair sample should contain about half from each, although the early items had far more chances of being thrown out again.

In [ ]:
sample = reservoir_sample(range(1, 100_001), k=10_000, seed=1)
print(f"From the first half of the stream:  {sum(1 for value in sample if value <= 50_000):,}")
print(f"From the second half of the stream: {sum(1 for value in sample if value > 50_000):,}")

### Formal definition
After $n$ items, every item seen so far is in the reservoir with the same probability, $k/n$. The argument is short. Item number $n$ enters with probability $k/n$ by construction. An item already in the reservoir stays unless the newcomer is admitted **and** happens to be given its place, so it survives with probability $1 - \frac{k}{n}\cdot\frac{1}{k} = \frac{n-1}{n}$. If it was in the reservoir with probability $\frac{k}{n-1}$ before, it is there with probability $\frac{k}{n-1}\cdot\frac{n-1}{n} = \frac{k}{n}$ afterwards.

How good is an estimate from a sample? That depends on the size of the **sample**, and hardly at all on the size of the data. A sample of 2,000 estimates a median about as well from a billion rows as from a million. This is why opinion polls of a thousand people can describe a country of sixty million.

## 5. Sorting a file that does not fit

### Intuition first
Sometimes an approximation will not do: we need the whole file sorted. Sorting seems to require having everything in hand at once.

Think of sorting an enormous pile of exam papers by name on a small desk. You cannot spread them all out. But you can take one desk-sized bundle at a time, sort it, and set it aside as a sorted stack. When every bundle is sorted, you **merge** the stacks: look only at the top paper of each, take whichever comes first alphabetically, and repeat. The desk never holds more than one bundle, and during the merge it holds only one paper from each stack.

This is the **external merge sort**, and every database uses it when asked to sort more than fits in memory.

### Formal definition
1. **Split and sort.** Read the input in chunks that fit in memory. Sort each chunk and write it to its own temporary file, called a *run*.
2. **Merge.** Open all the runs together and repeatedly output the smallest of their current first items.

The merge needs to remember one item per run. Python's `heapq.merge` does precisely this, and returns a stream.

In [ ]:
def external_sort(input_path, output_path, key_column, chunk_lines=50_000):
    def sort_key(line):
        return float(line.split(",")[key_column])

    run_paths = []
    with open(input_path, encoding="utf-8") as file:
        header = file.readline()
        while True:
            chunk = [line for line in (file.readline() for i in range(chunk_lines)) if line]      # one bundle
            if not chunk:
                break
            chunk.sort(key=sort_key)                                                              # sort it on the desk
            run_path = f"{output_path}.run{len(run_paths)}"
            with open(run_path, "w", encoding="utf-8") as run:
                run.writelines(chunk)                                                             # set it aside
            run_paths.append(run_path)

    runs = [open(path, encoding="utf-8") for path in run_paths]
    with open(output_path, "w", encoding="utf-8") as output:
        output.write(header)
        output.writelines(heapq.merge(*runs, key=sort_key))                                       # merge the stacks
    for run in runs:
        run.close()
    return len(run_paths)

sorted_path = os.path.join(folder, "sales_by_revenue.csv")
number_of_runs, elapsed_external, peak_external = measure(lambda: external_sort(sales_path, sorted_path, key_column=5))

print(f"Sorted using {number_of_runs} runs, in {elapsed_external:.1f} seconds, peak memory {peak_external:.1f} MB")
with open(sorted_path, encoding="utf-8") as file:
    lines = file.readlines()
print("Cheapest line:  ", lines[1].strip())
print("Dearest line:   ", lines[-1].strip())
print("Correctly sorted:", all(float(a.split(",")[5]) <= float(b.split(",")[5]) for a, b in zip(lines[1:], lines[2:])))

With the file sorted, the exact median is simply the line in the middle. We used the **disk** as the memory we did not have.

For comparison, the ordinary way: read everything, sort it in memory, write it out.

In [ ]:
def sort_in_memory(input_path, output_path, key_column):
    with open(input_path, encoding="utf-8") as file:
        header = file.readline()
        lines = file.readlines()
    lines.sort(key=lambda line: float(line.split(",")[key_column]))
    with open(output_path, "w", encoding="utf-8") as output:
        output.write(header)
        output.writelines(lines)

nothing, elapsed_memory, peak_memory = measure(lambda: sort_in_memory(sales_path, os.path.join(folder, "in_memory.csv"), 5))

print(f"{'':<16} {'time':>8} {'peak memory':>13}")
print(f"{'In memory':<16} {elapsed_memory:>6.1f} s {peak_memory:>10.1f} MB")
print(f"{'External sort':<16} {elapsed_external:>6.1f} s {peak_external:>10.1f} MB")

Look at both columns. The external sort used a fraction of the memory, and it was **slower**. It wrote every line to disk twice and compared items one at a time in the merge.

| | The data fits in memory | The data does not fit |
|---|---|---|
| Sort in memory | fast and simple | fails |
| External sort | works, and is slower for no benefit | the only one that works |
| Better choice | **in memory** | **external** |

The memory of the external sort is set by the size of the chunk, a number **we** choose, and not by the size of the file. That is the entire point. It is not a faster method. It is a method that keeps working.

## 6. The same idea in a library

Professional tools read large files in **batches**: not one row at a time, which is slow in Python, and not everything at once, but a few thousand rows per step. PyArrow reads CSV and Parquet this way. Here the CSV file is converted to Parquet in a stream, a batch at a time, so the conversion would work on a file of any size.

In [ ]:
import pyarrow as pa
import pyarrow.csv as pacsv
import pyarrow.parquet as pq
import pyarrow.compute as pc

parquet_path = os.path.join(folder, "sales.parquet")

reader = pacsv.open_csv(sales_path)                       # opens the file, reads nothing yet
writer = pq.ParquetWriter(parquet_path, reader.schema)
batches = 0
for batch in reader:                                      # one batch of rows at a time
    writer.write_batch(batch)
    batches += 1
writer.close()

print(f"Converted in {batches} batches")
print(f"CSV: {os.path.getsize(sales_path) / 1e6:.1f} MB   Parquet: {os.path.getsize(parquet_path) / 1e6:.1f} MB")

And the grouped total of Section 3, reading the Parquet file in batches and asking only for the two columns needed, as in Notebook 05.

In [ ]:
totals = {}
for batch in pq.ParquetFile(parquet_path).iter_batches(batch_size=50_000, columns=["category", "revenue"]):
    grouped = pa.Table.from_batches([batch]).group_by("category").aggregate([("revenue", "sum")])      # a small result per batch
    for category, revenue in zip(grouped["category"].to_pylist(), grouped["revenue_sum"].to_pylist()):
        totals[category] = totals.get(category, 0.0) + revenue                                         # combine the small results

for category, revenue in sorted(totals.items(), key=lambda item: -item[1])[:4]:
    print(f"{category:<10} {revenue:>14,.2f}")

Look closely at the shape of that loop, because you are about to meet it again at a much larger scale. Each batch is summarised **on its own** into a small table of totals. The small tables are then **combined** by adding. No step ever sees the whole dataset.

Nothing in that recipe requires the batches to be processed one after another, or on the same machine. Hand each batch to a different computer, and combine what they send back. That is the idea of the next notebook.

## 7. Where we stand

| The problem | The answer |
|---|---|
| A Python object is far larger than its text on disk | measure before assuming that data fits |
| Loading a file needs memory in proportion to its size | stream it, and keep only a small state |
| The answer itself is small (a total, a grouping) | compute it in one pass |
| The answer needs every value (a median) | estimate it from a fair sample of the stream |
| Everything must be sorted | external merge sort: use the disk |

The data stopped fitting in **memory**, and for many purposes it did not matter. A single ordinary machine, used carefully, can process far more data than it can hold. Remember this when someone proposes a cluster for a file of a few gigabytes.

There is one thing streaming cannot do. It makes the work **possible**, and it does not make it **faster**. One machine reading a hundred terabytes one row at a time will finish, eventually. When "eventually" is too long, the only way forward is to put more machines to work at once.

---
## ✏️ Exercises

### Exercise 1 (Sizes)
Use `sys.getsizeof` to find the size in bytes of: the integer `1`, the integer `10**30`, the empty string, a string of 100 letters, an empty list, and a list of 1,000 `None` values. Which grow with their content? What does the last result tell you about what a list actually stores?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for label, value in [("1", 1), ("10**30", 10**30), ("empty string", ""), ("100 letters", "a" * 100),
                     ("empty list", []), ("1,000 None values", [None] * 1000)]:
    print(f"{label:<20} {sys.getsizeof(value):>6} bytes")
```

*The list of a thousand items takes about 8,000 bytes more than the empty one: eight bytes per item. A list does not contain its items. It contains references to them, each the size of a memory address, and the items live elsewhere. That is why `sys.getsizeof` of a list of rows tells you very little, and why we measured with `tracemalloc`.*
</details>

### Exercise 2 (One pass)
Write a function `busiest_day(rows)` that takes the stream from `read_sales` and returns the date with the most order lines, with its count, in one pass. How large can its state become?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def busiest_day(rows):
    counts = {}
    for row in rows:
        counts[row["date"]] = counts.get(row["date"], 0) + 1
    day = max(counts, key=counts.get)
    return day, counts[day], len(counts)

(day, count, groups), elapsed, peak = measure(lambda: busiest_day(read_sales(sales_path)))
print(f"{day}: {count} lines.  State: {groups} entries, peak memory {peak:.2f} MB")
```

*The state has one entry per distinct date, 730 here, however many million rows go by. The data could grow for ever and the state would grow only by 365 entries a year.*
</details>

### Exercise 3 (A variance without keeping the values)
The variance of a list is the average of the squared differences from the mean. That seems to need two passes: one for the mean, one for the differences. Show that one is enough: accumulate the count, the sum of the values and the sum of their **squares**, and use $\text{variance} = \frac{\sum x^2}{n} - \left(\frac{\sum x}{n}\right)^2$. Compute the standard deviation of the revenue in a single pass and compare with `statistics.pstdev` on a fully loaded list.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
import statistics

def streaming_deviation(values):
    count, total, total_of_squares = 0, 0.0, 0.0
    for value in values:
        count += 1
        total += value
        total_of_squares += value * value
    mean = total / count
    return (total_of_squares / count - mean * mean) ** 0.5

print("One pass, three numbers of state:", round(streaming_deviation(field(read_sales(sales_path), "revenue")), 4))
print("With every value in memory:      ", round(statistics.pstdev(list(field(read_sales(sales_path), "revenue"))), 4))
```

*Three numbers of state are enough. A quantity that can be computed from a few running totals is called decomposable, and it has a second valuable property: two machines can each compute their own three totals, and the results can simply be added. Sums, counts, averages and variances all work this way. Medians do not.*
</details>

### Exercise 4 (How big should the sample be?)
Estimate the median revenue from reservoir samples of 10, 100, 1,000 and 10,000 values. For each size, repeat the estimate with 20 different seeds and print the smallest and largest estimate obtained. How does the spread change when the sample is ten times larger?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
revenues = list(field(read_sales(sales_path), "revenue"))        # loaded once here, only to make the experiment quick
true_median = sorted(revenues)[len(revenues) // 2]

print(f"True median: {true_median:.2f}\n")
print(f"{'sample size':>11} {'lowest estimate':>16} {'highest estimate':>17} {'spread':>8}")
for k in [10, 100, 1000, 10000]:
    estimates = []
    for seed in range(20):
        sample = reservoir_sample(revenues, k, seed=seed)
        estimates.append(sorted(sample)[len(sample) // 2])
    print(f"{k:>11,} {min(estimates):>16.2f} {max(estimates):>17.2f} {max(estimates) - min(estimates):>8.2f}")
```

*Each tenfold increase in the sample narrows the spread by a factor of about three, the square root of ten. Accuracy improves with the square root of the sample size. Halving the error costs four times the sample, which is why very precise answers are expensive and rough ones are cheap.*
</details>

### Exercise 5 (Smaller desk, a little harder)
Run `external_sort` with `chunk_lines` set to 5,000, 20,000, 100,000 and 400,000, measuring each with `measure`. Print the number of runs, the time and the peak memory. What does the last setting correspond to? Is there a setting that is best in every respect?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
print(f"{'chunk':>8} {'runs':>5} {'time':>8} {'peak memory':>13}")
for chunk_lines in [5_000, 20_000, 100_000, 400_000]:
    runs, elapsed, peak = measure(lambda: external_sort(sales_path, os.path.join(folder, "trial.csv"), 5, chunk_lines))
    print(f"{chunk_lines:>8,} {runs:>5} {elapsed:>6.1f} s {peak:>10.1f} MB")
```

*Memory rises in step with the chunk size, since the chunk is what must fit on the desk. A chunk of 400,000 lines holds the whole file: one run, and the "external" sort has become an in-memory sort. There is no best setting, only a budget: choose the largest chunk that fits comfortably in the memory you have. Very small chunks also have a cost, because the merge must then keep hundreds of files open at once.*
</details>

## 🔑 Key takeaways

- Data in memory is **much larger** than the same data on disk. Measure, do not guess.
- A **stream** is processed one item at a time, keeping only a small state. Its memory does not grow with the input.
- **Generators** (`yield`) build streams, and chains of them form pipelines.
- Totals, averages and groupings can be computed in **one pass**, when the **result** is small.
- **Reservoir sampling** draws a fair sample from a stream of unknown length. The accuracy of an estimate depends on the size of the sample, not of the data.
- **External merge sort** sorts more than fits in memory by sorting chunks and merging them from disk.
- Streaming makes work **possible**, not faster. When the data fits, loading it is simpler and quicker.

---
**Next:** *Notebook 08: MapReduce by Hand*, where the pattern at the end of this notebook, summarise each piece separately and combine the results, is given to many workers at the same time.

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*